In [ ]:
# pd.set_option("future.no_silent_downcasting", True)
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import argparse
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex
import yaml

from src.api import cargarHistorico, getHistoricoMOW
from src.processor import LogProcessor
from src.utils import isValidCode, parallelizeFunction, parseDate, rellenarId

In [ ]:
# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "MANIOBRALLEGADA",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "EXIT",
            "MANIOBRASALIDA",
            "MANIOBRA",
        ]
    )
}


def processOcupacion(
    df_logs: pd.DataFrame,
    save_info: bool,
    save_dir: Path,
    days: str,
    estaciones=list[str],
    list_rotaciones: list = [],
    desc: str = "Procesando estaciones",
):
    log_processor = LogProcessor()
    used_dfs = parallelizeFunction(
        log_processor.processStation,
        estaciones,
        df=df_logs,
        mov_sorter=mov_sorter,
        save_info=save_info,
        save_dir=save_dir,
        days=days,
        list_rotaciones=list_rotaciones,
        leave=True,
        desc=f"{desc}: {days}",
    )
    # with tqdm(total=len(estaciones), position=0, leave=False) as pbar:
    #     for e in estaciones:
    #         pbar.set_description(f"Procesando '{e}'")
    #         used_dfs = log_processor.processStation(
    #             e,
    #             df=df_logs,
    #             mov_sorter=mov_sorter,
    #             save_info=save_info,
    #             save_dir=save_dir,
    #             days=days,
    #             list_rotaciones=list_rotaciones,
    #         )
    #         pbar.update()
    return used_dfs

In [ ]:
# Cargamos el fichero de rotaciones
rot = pd.read_csv("data/20240112_fichero_rotacion.txt", sep="\t")
rot["Fecha"] = pd.to_datetime(rot["Fecha"], format="%Y%m%d")
rot[["T.ida", "T.vuelta"]] = rot[["T.ida", "T.vuelta"]].map(rellenarId)
day = pd.to_datetime("2024-01-15")
rotaciones = rot[rot["Fecha"] == day].drop_duplicates().dropna()
list_rotaciones = [tuple(el) for el in rotaciones[["T.ida", "T.vuelta"]].values]

In [ ]:
start_date = "2025-04-24"
end_date = "2025-04-24"
start_date = parseDate(start_date)
end_date = parseDate(end_date)

estaciones = [
    # "60000",
    "54413",
]
trenes = []


# Cargamos histórico
df_logs = cargarHistorico(start_date, end_date, estaciones, trenes)

# Procesamos y guardamos
d1 = regex.sub(r"[-:\s]", "", f"{pd.to_datetime(start_date).date()}")
d2 = regex.sub(r"[-:\s]", "", f"{pd.to_datetime(end_date).date()}")
used_dfs = processOcupacion(
    df_logs,
    False,
    None,
    f"{d1} - {d2}",
    estaciones,
    list_rotaciones,
)

In [ ]:
canon_cols = {
    "Producto": "TIPO TREN",
    "T1": "TREN ENTRA",
    "T2": "TREN SALE",
    "InicioOcupación": "ENTRADA",
    "FinOcupación": "SALIDA",
    "EF": "EmpresaFerroviaria",
    "Vía": "Vía",
    "Movimiento": "Movimiento",
    "Ocupación": "Ocupación",
}
tipo_tren = {
    "CERCANIAS": "CERCANIAS",
    "AVE": "AVMD",
    "AVANT": "AVMD",
    "IRYO": "AVMD",
    "MD": "AVMD",
    "OUIGO": "AVMD",
    "Material Vacio": "",
}

In [ ]:
used_dfs[0][1]

In [ ]:
df = used_dfs[0][0].copy()
hoja_canon = df[list(canon_cols.keys())].rename(columns=canon_cols).copy()
hoja_canon["TIPO TREN"] = hoja_canon["TIPO TREN"].apply(tipo_tren.get)
hoja_canon = hoja_canon[hoja_canon["TIPO TREN"] == "AVMD"]
hoja_canon.sort_values(by="TREN ENTRA").head(20)

In [ ]:
df_logs[df_logs["NTécnico"]=="10193"]

In [ ]:
df.loc[
    df["T_seq"].apply(lambda x: bool(regex.search("(02142|10193)", x))),
    [
        "T1",
        "T2",
        "T_seq",
        "ProductoT1",
        "ProductoT2",
        "EF",
        "Vía",
        "Movimiento",
        "Mov_seq",
        "full_seq",
        "cambio_vía",
        "InicioOcupación",
        "FinOcupación",
        "Ocupación",
        "TipoVía",
        "Producto",
    ],
]